# 欢迎来到你的第一个任务！

说明如下。请尝试一下，如果遇到困难，请查看解决方案文件夹（或者随时问我！）

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">就在我们开始作业之前 --</h2>
            <span style="color:#f71;">我想花点时间向您介绍本课程的有用资源页面。这包括所有幻灯片的链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            请保留此书签，随着时间的推移，我将继续添加更多有用的链接。
            </span>
        </td>
    </tr>
</表>

# 家庭作业练习作业

升级第 1 天项目以总结网页，以使用通过 Ollama 而不是 OpenAI 在本地运行的开源模型

如果您不想使用付费 API，您将能够在所有后续项目中使用此技术。

**好处：**
1.无API收费——开源
2. 数据不会离开你的盒子

**缺点：**
1. 功耗明显低于 Frontier 模型

## Ollama 安装回顾

只需访问 [ollama.com](https://ollama.com) 并安装即可！

完成后，ollama 服务器应该已经在本地运行。  
如果您访问：  
[http://localhost:11434/](http://localhost:11434/)

您应该看到消息“Ollama 正在运行”。  

如果没有，请打开新的终端 (Mac) 或 Powershell (Windows) 并输入 `ollamaserve`  
在另一个终端 (Mac) 或 Powershell (Windows) 中，输入 `ollama pull llama3.2`  
然后再次尝试 [http://localhost:11434/](http://localhost:11434/)。

如果 Ollama 在您的计算机上运行缓慢，请尝试使用“llama3.2:1b”作为替代方案。从终端或 Powershell 运行 `ollama pull llama3.2:1b`，并将下面的代码从 `MODEL = "llama3.2"` 更改为 `MODEL = "llama3.2:1b"`

In [ ]:
# 导入

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [ ]:
# 常量

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [ ]:
# 使用与 OpenAI 相同的格式创建消息列表

messages = [
    {"role": "user", "content": "Describe some of the business applications of Generative AI"}
]

In [ ]:
payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False
    }

In [ ]:
# 让我们确保模型已加载

!ollama pull llama3.2

In [ ]:
# 如果由于任何原因这不起作用，请尝试以下单元格中的 2 个版本
# 并仔细检查本实验顶部“Ollama 安装回顾”中的说明
# 如果这些都不起作用 - 联系我！

response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
print(response.json()['message']['content'])

# 介绍 ollama 包

现在我们将做同样的事情，但使用优雅的 ollama python 包而不是直接的 HTTP 调用。

在幕后，它对在 localhost:11434 运行的 ollama 服务器进行与上面相同的调用

In [ ]:
import ollama

response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

## 替代方法 - 使用 OpenAI python 库连接到 Ollama

In [ ]:
# 实际上，有些人可能更喜欢另一种方法
# 您可以使用OpenAI客户端python库来调用Ollama：

from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

response = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

## 您是否对为什么会这样感到困惑？

看起来很奇怪，对吧？我们刚刚使用OpenAI代码来调用Ollama？？这是怎么回事？！

这是独家新闻：

Python 类“OpenAI”只是由 OpenAI 工程师编写的代码，用于通过互联网向端点进行调用。  

当您调用 `openai.chat.completions.create()` 时，此 Python 代码仅向以下 url 发出 Web 请求：“https://api.openai.com/v1/chat/completions”

这样的代码被称为“客户端库”——它只是在您的计算机上运行以发出 Web 请求的包装代码。 GPT 的真正威力在于该 API 背后的 OpenAI 云上运行，而不是在您的计算机上！

OpenAI 非常受欢迎，以至于许多其他 AI 提供商都提供了相同的 Web 端点，因此您可以使用相同的方法。

因此，Ollama 有一个端点在您的本地机器上运行，地址为 http://localhost:11434/v1/chat/completions  
在第 2 周，我们会发现许多其他提供商也这样做，包括 Gemini 和 DeepSeek。

然后 OpenAI 的团队有了一个好主意：他们可以扩展他们的客户端库，这样你就可以指定不同的“基本 url”，并使用他们的库来调用任何兼容的 API。

就是这样！

所以当你说：`ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
然后，这将进行相同的端点调用，但调用的是 Ollama 而不是 OpenAI。

## 还尝试了令人惊叹的推理模型 DeepSeek

这里我们使用已精简至 1.5B 的 DeepSeek-reasoner 版本。  
这实际上是 Qwen 的 1.5B 变体，已使用 Deepseek R1 生成的 Synethic 数据进行了微调。

DeepSeek 的其他尺寸[此处](https://ollama.com/library/deepseek-r1) 一直到完整的 671B 参数版本，这将占用您的驱动器 404GB，对于大多数人来说太大了！

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
# 这可能需要几分钟才能运行！然后，您应该在 <think> 标签内看到令人着迷的“思考”痕迹，后面是一些不错的定义

response = ollama_via_openai.chat.completions.create(
    model="deepseek-r1:1.5b",
    messages=[{"role": "user", "content": "Please give definitions of some core concepts behind LLMs: a neural network, attention and the transformer"}]
)

print(response.choices[0].message.content)

# 现在是你的练习

将第一天的代码合并到这里，构建一个使用本地运行的 Llama 3.2 而不是 OpenAI 的网站摘要器；使用上述方法之一。

In [ ]:
import os
import requests
# 从 dotenv 导入 load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI

In [ ]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
webpage = Website("https://www.pleasurewebsite.com")
print(webpage.title)
print(webpage.text)

In [ ]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [ ]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [ ]:
messages=messages_for(webpage)

In [ ]:
import ollama
MODEL = "llama3.2"
response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])